# Assignment 3 — Bridge structural monitoring

You have been hired to monitor the pedestrian suspension bridge Ponte Tibetano Carasc, between Monte Carasso and Sementina. The objective is to identify abnormal vibration and deformation patterns that may occur during strong wind or unusual dynamic loading. The node is fixed near the bridge deck and simulates a structural health monitoring device combining an accelerometer, a deck displacement sensor and a temperature probe.

**Location:** Ponte Tibetano Carasc, Monte Carasso–Sementina, Canton Ticino, Switzerland  
**Thing to register in istSOS4:** `Bridge structural node CARASC-BRG-01`  
**Sensor:** Structural health monitoring node with accelerometer, displacement sensor and cable temperature probe


## Assignment

Design and register a SensorThings/istSOS4 monitoring setup for this case. Then run the real-time generator and send the simulated observations to the correct Datastreams.

Each generated row has this form:

```python
[phenomenonTime, value_1, value_2, value_3]
```

The generator does **not** assign quality flags. Each value starts as raw data (`0`) and your code must classify it using the thresholds below.

| Quality | Meaning | Rule |
|---:|---|---|
| 0 | raw | value not yet checked |
| 1 | sensible | value inside statistical thresholds |
| 2 | suspect | value outside statistical thresholds but still inside plausibility thresholds |
| 3 | alarm | value outside plausibility thresholds |
| -999 | invalid | value outside physical limits |

| Parameter | Meaning | Unit | Physical limit | Plausibility threshold | Statistical threshold |
|---|---|---:|---:|---:|---:|
| `vertical_acceleration_g` | Vertical dynamic acceleration of the deck | g | -3 – 3 | -1.5 – 1.5 | -0.25 – 0.25 |
| `deck_displacement_mm` | Deck displacement relative to a local reference | mm | -120 – 120 | -80 – 80 | -15 – 15 |
| `cable_temperature_c` | Cable or deck temperature | °C | -40 – 90 | -25 – 65 | -5 – 35 |



## Generator behaviour

The generator runs in real time when `sleep=True`. With `step_seconds=30`, one row is produced every 30 seconds.

Recommended settings for the assignment:

```python
alarm_after_seconds=600       # first alarm after about 10 minutes
alarm_duration_seconds=180    # alarm lasts about 3 minutes
alarm_repeat_seconds=None     # no repetition
```

If you want repeated events during a longer exercise, set for example:

```python
alarm_repeat_seconds=900
```

This means that, after the first alarm start, a new alarm window starts every 900 seconds. If it is left as `None`, the alarm happens only once.


In [ ]:
QUALITY_RAW = 0
QUALITY_SENSIBLE = 1
QUALITY_SUSPECT = 2
QUALITY_ALARM = 3


In [ ]:
from scripts.sensor_stream_generators import BridgeStructuralGenerator

generator = BridgeStructuralGenerator(
    step_seconds=30,
    alarm_after_seconds=600,
    alarm_duration_seconds=180,
    alarm_repeat_seconds=None,  # set e.g. 900 to repeat alarms every 15 minutes
    suspect_probability=0.08,
    seed=7
)

print(generator.cols())
for i in range(10):  # adjust the range as needed
    row = generator.read(sleep=False)  # use sleep=True in the real acquisition loop
    print(row)
    print("simulated status:", generator.status())


In [ ]:
# Example real-time acquisition loop.
# In the workshop, replace the print section with your POST requests to istSOS4.

for i in range(5):
    row = generator.read(sleep=False)  # change to sleep=True to wait 30 seconds between rows
    checked_observations = row_to_checked_observations(row, generator.cols(), parameters)
    print("row", i, row, "status=", generator.status())
    for obs in checked_observations:
        print("  ", obs)
        # TODO: push obs to the correct istSOS4 Datastream
